In [1]:
!python --version

Python 3.11.14


In [2]:
import os, sys
import pandas as pd
import requests
import time

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [3]:
ROOT_DATA = '../data'
ROOT_DATA = Path(ROOT_DATA)

gdc = GDC(ROOT_DATA0=ROOT_DATA)

os.listdir(ROOT_DATA)[:10]


['cancer', 'reactome', 'vector_store', 'TCGA', 'GTEx', 'gdc_programs.txt']

In [4]:
os.listdir(gdc.root_data)[:10]

['app_main_local.py',
 'gtex_normal_tissue.ipynb',
 'docker_config.ipynb',
 'gtex_normal_tissue_dvlp.ipynb',
 'tcga_gdc_run_R_expression_dvlp.ipynb']

### Get all programs

In [5]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

np.array(prog_list)


File read at '../data/gdc_programs.txt'


array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.self.cancer.gov/projects'

In [7]:
prog_id = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)
print(len(df_psi))

df_psi.head(3)

Table opened ((33, 5)) at '../data/TCGA/primary_site_program_TCGA.tsv'
33


,psi_id,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma


### GTEx

In [8]:
df_gtex = gdc.read_GTEx_to_TCGA_table()
print(len(df_gtex))

df_gtex.head(3)

33


,tcga_project_id,tcga_primary_site,gtex_dataset_id,preferred_gtex_id,gtex_tissue_site_detail_ids,mapping_confidence,notes,source_gtex_api
0,TCGA-ACC,Adrenal Gland,gtex_v8,Adrenal_Gland,Adrenal_Gland,high,Adrenocortical carcinoma; direct adrenal gland...,https://gtexportal.org/api/v2/redoc
1,TCGA-BLCA,Bladder,gtex_v8,Bladder,Bladder,high,"Direct bladder GTEx tissue, but GTEx bladder s...",https://gtexportal.org/api/v2/redoc
2,TCGA-BRCA,Breast,gtex_v8,Breast_Mammary_Tissue,Breast_Mammary_Tissue,high,Breast carcinoma; GTEx mammary/breast tissue.,https://gtexportal.org/api/v2/redoc


In [9]:
verbose = True

psi_id = 'TCGA-BRCA'
psi_id = 'TCGA-LUAD'

gtex_id = gdc.find_GTEx_to_TCGA_row(psi_id=psi_id, verbose=verbose)
print(">>>", gtex_id)


Found 'Lung' tissue 'Lung'
>>> Lung


### Get df_tumor - entropy - to find homogeneous rows

In [10]:
verbose=False

df_tumor, df_normal = gdc.get_file_expression_both_tumor_and_normal(psi_id=psi_id, verbose=verbose)

len(df_tumor), len(df_normal)

>>> TCGA-LUAD
>>> TCGA-LUAD
0.........10.........20.........30.........40.........50.........60.........70.........80.........90......... -> 100
Dowloading tumor files: 0.........10.........20.........30.........40.........50.........60.........70.........80.........90.........100........ -> 109
>>> Processing normal data: 100
1 2 3 4 5 6 7 8 9 10 11 12 13 
>>> Processing tumor data: 11
1 2 3 4 5 6 7 8 9 10 11 

(150684, 240796)

In [11]:
df_tumor.head(3)

,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11
0,ENSG00000000003,TSPAN6,protein_coding,1604,2392,1373,803,799,5095,2,792,1489,3870,522
1,ENSG00000000005,TNMD,protein_coding,0,0,8,2,1,0,14,0,0,1,0
2,ENSG00000000419,DPM1,protein_coding,459,816,713,799,400,3372,39,1043,345,1380,678


In [12]:
df_tumor2 = gdc.add_entropy(df_tumor)
print(len(df_tumor2))
df_tumor2.head(3)


15817


,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,total_sum,entropy,entropy_q
0,ENSG00000000003,TSPAN6,protein_coding,1604,2392,1373,803,799,5095,2,792,1489,3870,522,18741,2.954896,5
2,ENSG00000000419,DPM1,protein_coding,459,816,713,799,400,3372,39,1043,345,1380,678,10044,2.911364,4
3,ENSG00000000457,SCYL3,protein_coding,542,432,557,699,264,995,168,352,283,1706,471,6469,2.852843,3


In [13]:
tumor_cols = [c for c in df_tumor.columns if c.startswith("tumor_")]

df_tumor3 = gdc.select_top_entropy(df_tumor2, q=0.1)
print(len(df_tumor3))
df_tumor3.head(3)

1582


,gene_id,symbol,gene_type,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,total_sum,entropy,entropy_q
0,ENSG00000135919,SERPINE2,protein_coding,189,821,1026,2040,1170,216535,5,599,193,1394,754,224726,0.329341,0
1,ENSG00000235162,C12orf75,protein_coding,627,56,231,253,47,22188,52,620,113,203,70,24460,0.563498,0
2,ENSG00000235174,RPL39P3,processed_pseudogene,73,87,72,89,12,13687,0,213,11,663,58,14965,0.612907,0


In [14]:
geneid_list = list(np.unique(df_tumor3.gene_id))
len(geneid_list)

1018

### GTEx portal - data manually downloaded

In [15]:
print(len(geneid_list))
np.array(geneid_list[:50])

1018


array(['ENSG00000001626', 'ENSG00000003989', 'ENSG00000004468',
       'ENSG00000005020', 'ENSG00000005059', 'ENSG00000006007',
       'ENSG00000006016', 'ENSG00000006210', 'ENSG00000006534',
       'ENSG00000006747', 'ENSG00000007402', 'ENSG00000008226',
       'ENSG00000008735', 'ENSG00000008853', 'ENSG00000012171',
       'ENSG00000017483', 'ENSG00000018236', 'ENSG00000019102',
       'ENSG00000019186', 'ENSG00000021300', 'ENSG00000021826',
       'ENSG00000022267', 'ENSG00000022556', 'ENSG00000023171',
       'ENSG00000023839', 'ENSG00000025423', 'ENSG00000025708',
       'ENSG00000026103', 'ENSG00000026508', 'ENSG00000029153',
       'ENSG00000034053', 'ENSG00000038002', 'ENSG00000040531',
       'ENSG00000043039', 'ENSG00000047457', 'ENSG00000047662',
       'ENSG00000047936', 'ENSG00000048052', 'ENSG00000049247',
       'ENSG00000051128', 'ENSG00000053918', 'ENSG00000057294',
       'ENSG00000060718', 'ENSG00000060982', 'ENSG00000062038',
       'ENSG00000064199', 'ENSG000000642

In [16]:
gdc.gtex_id

'Lung'

In [17]:
verbose=False
force=False

want_run = False

if want_run:
    df_gtex = gdc.get_gtex_TPM_expression_for_geneid_list(
                    geneid_list=geneid_list,
                    datasetId="gtex_v8", page_size=1000,
                    sleep = 0.1, timeout=60,
                    force=force, verbose=verbose)
else:
    df_gtex = pd.DataFrame()

print(df_gtex.shape)

(0, 0)


### GTEx download

https://www.gtexportal.org/home/downloads/adult-gtex/overview

In [18]:
url_counts = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.gct.gz"
)

url_tpm = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.gct.gz"
)

url_meta = (
    "https://storage.googleapis.com/adult-gtex/metadata/"
    "GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
)


filename_counts = gdc.root_gtex / gdc.fname_GTEx_counts
filename_meta   = gdc.root_gtex / gdc.fname_GTEx_meta
filename_pheno = gdc.root_gtex / gdc.fname_GTEx_pheno

# --> download manually at https://www.gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression
#                      and https://www.gtexportal.org/home/downloads/adult-gtex/metadata

# gdc.download_file(url_counts, filename_counts)
# gdc.download_file(url_meta, filename_meta)

### Reading GTEx count super-file

In [ ]:
verbose=False

dfcounts = gdc.read_GTEx_counts(verbose=verbose)
print(len(dfcounts))
dfcounts.head(3)

### Phenotyp - DTHHRDY (Hardy Scale of Death)

> It indicates how the donor died and how fast, which affects tissue quality.  


| Value	| Meaning  |
|---------|-----------|
| 0	| Ventilator case (on life support before death) |
| 1	| Violent and fast death (<10 min) |
| 2	| Fast death (10 min – 1 hour) |
| 3	| Intermediate death (1 – 24 hours) |
| 4	| Slow death (>24 hours, prolonged illness) |
| NA	| Unknown |


In [ ]:
df_pheno = gdc.read_GTEx_pheno(verbose=verbose)

print(len(df_pheno))
df_pheno.head(3)

In [ ]:
df_meta = gdc.read_GTEx_metadata(verbose=verbose)

print(len(df_meta))
df_meta.head(3)

### Healthy donors

1. Filter the tissue
2. Filter for high quality (Hardy Scale 1 or 2 are 'fast' deaths, less stress)
  . DTHHRDY: 1 = Ventilator, 2 = Fast death of natural causes
3. Sort by RIN score (SMRIN) to get the best preserved RNA

In [ ]:
df_ctrl = gdc.prepare_df_control()

print(df_ctrl.shape)
df_ctrl.head(5)

In [ ]:
force=False
verbose=True

df_normal = gdc.prepare_count_table(Nsamples=15, force=force, verbose=verbose)
